Script 1

In [ ]:
# ==============================================================================
# SCRIPT 1: Advanced Feature Engineering Engine (Includes DD/TD Targets)
# ==============================================================================
# Adds the binary target variables for Double-Double and Triple-Double to the
# final training dataset.

# Step 0: Install necessary packages
!pip install pandas unidecode tqdm scikit-learn

import pandas as pd
import numpy as np
import os
import re
from datetime import datetime
from google.colab import drive
from unidecode import unidecode
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# --- Configuration ---
N_PLAYER_TRAITS = 10
ROLLING_WINDOWS = [5, 10, 20]
MODERN_ERA_START_SEASON = '2015-16'

# --- Helper Functions (omitted for brevity, they are the same as before) ---
def get_current_nba_season():
    simulated_today = datetime(2025, 11, 10)
    year = simulated_today.year; month = simulated_today.month
    return f"{year}-{str(year+1)[-2:]}" if month >= 9 else f"{year-1}-{str(year)[-2:]}"
def normalize_player_name(name):
    if not isinstance(name, str): return None
    name = unidecode(name.lower()); name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name); name = re.sub(r'[^\w\s]', '', name); name = re.sub(r'\s+', ' ', name).strip()
    return name
def convert_min_to_decimal(min_str):
    if isinstance(min_str, str) and ':' in min_str:
        try:
            minutes, seconds = map(int, min_str.split(':')); return minutes + seconds / 60.0
        except (ValueError, TypeError): return np.nan
    return pd.to_numeric(min_str, errors='coerce')

def derive_player_traits(df_for_clustering):
    print("\nDeriving player traits using K-Means Clustering...")
    stats_per_min = df_for_clustering[df_for_clustering['MIN'] > 5].copy()
    if stats_per_min.empty:
        print("Warning: Not enough player data with MIN > 5 to generate profiles for this season.")
        return pd.DataFrame()
    stat_cols = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'FGA', 'FG3A', 'FTA']
    for col in stat_cols:
        stats_per_min[f'{col}_PM'] = stats_per_min[col] / stats_per_min['MIN']
    player_profiles = stats_per_min.groupby(['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME'])[[f'{col}_PM' for col in stat_cols]].mean().reset_index()
    if player_profiles.empty: return pd.DataFrame()
    features_for_clustering = player_profiles[[f'{col}_PM' for col in stat_cols]]
    scaler = StandardScaler(); scaled_features = scaler.fit_transform(features_for_clustering)
    pca = PCA(n_components=0.95); reduced_features = pca.fit_transform(scaled_features)
    kmeans = KMeans(n_clusters=N_PLAYER_TRAITS, random_state=42, n_init='auto'); player_profiles['TRAIT'] = kmeans.fit_predict(reduced_features)
    cluster_analysis = player_profiles.groupby('TRAIT')[[f'{col}_PM' for col in stat_cols]].mean()
    print("\n--- Player Trait Cluster Analysis ---"); print(cluster_analysis)
    return player_profiles[['SEASON_YEAR', 'PLAYER_ID', 'TRAIT']]

def process_nba_data_advanced():
    """Main function to load data, engineer advanced features, and save."""
    try:
        print("Mounting Google Drive..."); drive.mount('/content/drive', force_remount=True)
        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data/"
        BOXSCORE_DIR = os.path.join(DATA_DIR, "NocturneBear_Boxscores")
        os.makedirs(DATA_DIR, exist_ok=True)
    except Exception as e: print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    print("\nStep 1: Loading and cleaning data...")
    file_paths = [os.path.join(BOXSCORE_DIR, f) for f in os.listdir(BOXSCORE_DIR) if f.endswith('.csv')]
    df = pd.concat([pd.read_csv(path, low_memory=False) for path in file_paths], ignore_index=True)
    df.rename(columns={'season_year': 'SEASON_YEAR', 'game_date': 'GAME_DATE', 'gameId': 'GAME_ID', 'personId': 'PLAYER_ID', 'personName': 'PLAYER_NAME', 'teamTricode': 'TEAM_ABBREVIATION', 'points': 'PTS', 'reboundsTotal': 'REB', 'assists': 'AST', 'minutes': 'MIN', 'fieldGoalsAttempted': 'FGA', 'freeThrowsAttempted': 'FTA', 'threePointersAttempted': 'FG3A', 'steals': 'STL', 'blocks': 'BLK'}, inplace=True)
    CURRENT_SEASON = get_current_nba_season()
    print(f"Simulated Current Season: {CURRENT_SEASON}")
    df = df[(df['SEASON_YEAR'] >= MODERN_ERA_START_SEASON) & (df['SEASON_YEAR'] < CURRENT_SEASON)].copy()
    print(f"Filtered training data to seasons: {df['SEASON_YEAR'].min()} to {df['SEASON_YEAR'].max()}")
    df = df[df['MIN'].notna() & (df['MIN'] != '0:00') & (df['MIN'] != '0') & (df.get('comment', pd.Series(dtype=str)).isna())].copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df['MIN'] = df['MIN'].apply(convert_min_to_decimal)
    # *** ADD STL and BLK to numeric cols for DD/TD calculation ***
    numeric_cols = ['PTS', 'REB', 'AST', 'MIN', 'FGA', 'FTA', 'FG3A', 'STL', 'BLK']
    for col in numeric_cols:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')
    df.dropna(subset=['MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK'], inplace=True)
    df.sort_values(by=['PLAYER_ID', 'GAME_DATE'], inplace=True)

    # (Steps 2, 3, 4 are unchanged - omitted for brevity)
    print("\nStep 2: Deriving Player Traits for each season...")
    all_seasons_traits = [];
    for season in sorted(df['SEASON_YEAR'].unique()):
        print(f"\n--- Processing season: {season} ---");
        season_df = df[df['SEASON_YEAR'] == season].copy();
        trait_map = derive_player_traits(season_df)
        if not trait_map.empty: all_seasons_traits.append(trait_map)
    player_trait_map = pd.concat(all_seasons_traits);
    trait_map_path = os.path.join(DATA_DIR, "player_trait_map.csv");
    player_trait_map.to_csv(trait_map_path, index=False);
    print(f"\nSaved player trait map to: {trait_map_path}");
    df = pd.merge(df, player_trait_map, on=['SEASON_YEAR', 'PLAYER_ID'], how='left')

    print("\nStep 3: Engineering Player Rolling Average and Trend Features...")
    stats_to_roll = ['PTS', 'REB', 'AST', 'MIN']; grouped = df.groupby('PLAYER_ID')
    for stat in tqdm(stats_to_roll, desc="Calculating player averages"):
        for w in ROLLING_WINDOWS: df[f'ROLL_AVG_{stat}_{w}'] = grouped[stat].transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
        df[f'TREND_{stat}'] = df[f'ROLL_AVG_{stat}_{ROLLING_WINDOWS[0]}'] - df[f'ROLL_AVG_{stat}_{ROLLING_WINDOWS[-1]}']
    df['DAYS_OF_REST'] = grouped['GAME_DATE'].transform(lambda x: x.diff().dt.days - 1).clip(upper=7).fillna(7)

    print("\nStep 4: Engineering Advanced DvP (Defense vs. Trait)...")
    opponent_df = df[['GAME_ID', 'TEAM_ABBREVIATION']].drop_duplicates();
    df = pd.merge(df, opponent_df.rename(columns={'TEAM_ABBREVIATION': 'OPPONENT_ABBR'}), on='GAME_ID');
    df = df[df['TEAM_ABBREVIATION'] != df['OPPONENT_ABBR']].copy()
    team_allowed_stats = df.groupby(['GAME_ID', 'GAME_DATE', 'OPPONENT_ABBR', 'TRAIT'])[['PTS', 'REB', 'AST']].sum().reset_index()
    team_allowed_stats.rename(columns={'OPPONENT_ABBR': 'TEAM'}, inplace=True);
    team_allowed_stats.sort_values(by=['TEAM', 'GAME_DATE'], inplace=True);
    dvp_grouped = team_allowed_stats.groupby(['TEAM', 'TRAIT'])
    for stat in ['PTS', 'REB', 'AST']: team_allowed_stats[f'OPP_DvT_{stat}'] = dvp_grouped[stat].transform(lambda x: x.shift(1).rolling(window=30, min_periods=5).mean())
    dvp_cols_to_merge = ['GAME_ID', 'TEAM', 'TRAIT'] + [c for c in team_allowed_stats if c.startswith('OPP_DvT')]
    df = pd.merge(df, team_allowed_stats[dvp_cols_to_merge], left_on=['GAME_ID', 'OPPONENT_ABBR', 'TRAIT'], right_on=['GAME_ID', 'TEAM', 'TRAIT'], how='left')

    print("\nStep 5: Finalizing data and saving...")
    # Regression Targets
    df['TARGET_MIN'] = df['MIN']; df['TARGET_PTS'] = df['PTS']; df['TARGET_REB'] = df['REB']; df['TARGET_AST'] = df['AST']
    df['TARGET_P+R'] = df['PTS'] + df['REB']; df['TARGET_P+A'] = df['PTS'] + df['AST']
    df['TARGET_R+A'] = df['REB'] + df['AST']; df['TARGET_PRA'] = df['PTS'] + df['REB'] + df['AST']

    # *** NEW: Classification Targets for DD/TD ***
    dd_td_stats = ['PTS', 'REB', 'AST', 'STL', 'BLK']
    df['DD_COUNT'] = df[dd_td_stats].ge(10).sum(axis=1)
    df['TARGET_DD'] = (df['DD_COUNT'] >= 2).astype(int)
    df['TARGET_TD'] = (df['DD_COUNT'] >= 3).astype(int)
    # ***********************************************

    feature_columns = [c for c in df.columns if 'ROLL_AVG' in c or 'TREND' in c or 'OPP_DvT' in c] + ['DAYS_OF_REST']
    target_columns = [c for c in df.columns if 'TARGET' in c]
    final_df = df[['GAME_DATE', 'PLAYER_ID', 'PLAYER_NAME'] + feature_columns + target_columns].copy()
    final_df.dropna(subset=[c for c in target_columns if c not in ['TARGET_DD', 'TARGET_TD']], inplace=True)
    final_df.dropna(subset=[c for c in feature_columns if 'OPP_DvT' in c], inplace=True)

    output_path = os.path.join(DATA_DIR, "nba_advanced_training_data.csv")
    final_df.to_csv(output_path, index=False)

    print(f"\n✅ Processing complete! {len(final_df)} model-ready rows saved to:")
    print(output_path)
    from IPython.display import display; display(final_df.head())

if __name__ == "__main__":
    process_nba_data_advanced()

Mounting Google Drive...
Mounted at /content/drive

Step 1: Loading and cleaning data...
Simulated Current Season: 2025-26
Filtered training data to seasons: 2015-16 to 2023-24

Step 2: Deriving Player Traits for each season...

--- Processing season: 2015-16 ---

Deriving player traits using K-Means Clustering...

--- Player Trait Cluster Analysis ---
         PTS_PM    REB_PM    AST_PM    STL_PM    BLK_PM    FGA_PM   FG3A_PM  \
TRAIT                                                                         
0      0.408147  0.160193  0.069260  0.031339  0.013188  0.351889  0.112177   
1      0.325818  0.282677  0.032774  0.019811  0.078431  0.253797  0.005048   
2      0.608713  0.142399  0.143439  0.036499  0.012201  0.485107  0.137585   
3      0.403707  0.087099  0.168010  0.140687  0.007848  0.385161  0.101752   
4      0.237053  0.135438  0.067467  0.031636  0.014736  0.242702  0.107060   
5      0.227670  0.288563  0.050866  0.017035  0.013851  0.202604  0.022321   
6      0.3548

Calculating player averages: 100%|██████████| 4/4 [00:06<00:00,  1.52s/it]



Step 4: Engineering Advanced DvP (Defense vs. Trait)...

Step 5: Finalizing data and saving...

✅ Processing complete! 225312 model-ready rows saved to:
/content/drive/MyDrive/NBA/Historical Data/nba_advanced_training_data.csv


,GAME_DATE,PLAYER_ID,PLAYER_NAME,ROLL_AVG_PTS_5,ROLL_AVG_PTS_10,ROLL_AVG_PTS_20,TREND_PTS,ROLL_AVG_REB_5,ROLL_AVG_REB_10,ROLL_AVG_REB_20,...,TARGET_MIN,TARGET_PTS,TARGET_REB,TARGET_AST,TARGET_P+R,TARGET_P+A,TARGET_R+A,TARGET_PRA,TARGET_DD,TARGET_TD
9,2015-11-17,708,Kevin Garnett,3.2,2.666667,2.666667,0.533333,5.0,4.777778,4.777778,...,14.716667,4,7,2,11,6,9,13,0,0
10,2015-11-20,708,Kevin Garnett,3.6,2.800000,2.800000,0.800000,5.4,5.000000,5.000000,...,13.650000,0,4,0,4,0,4,4,0,0
11,2015-11-23,708,Kevin Garnett,2.8,2.400000,2.545455,0.254545,4.6,4.700000,4.909091,...,17.733333,8,10,1,18,9,11,19,0,0
12,2015-11-25,708,Kevin Garnett,4.4,2.800000,3.000000,1.400000,5.6,5.100000,5.333333,...,14.983333,4,3,2,7,6,5,9,0,0
15,2015-12-01,708,Kevin Garnett,2.8,3.200000,2.800000,0.000000,5.6,5.500000,5.200000,...,20.416667,4,8,0,12,4,8,12,0,0


Script 2

In [ ]:
# ==============================================================================
# SCRIPT 2: Three-Stage LGBM Model Training Engine (with DD/TD)
# ==============================================================================
# Stage 1: Trains Minutes model.
# Stage 2: Trains Core Stats models.
# Stage 3: Trains DD/TD Classification models using projections from Stage 2.

!pip install lightgbm scikit-learn pandas

import pandas as pd
import numpy as np
import joblib
import os
import lightgbm as lgb
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score # Add roc_auc_score
from sklearn.impute import SimpleImputer

def train_and_evaluate_models_lgbm():
    """Trains all regression and classification models in a three-stage pipeline."""
    try:
        print("Mounting Google Drive..."); drive.mount('/content/drive', force_remount=True)
        DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data/"
        MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model Advanced"
        os.makedirs(MODEL_DIR, exist_ok=True)
    except Exception as e: print(f"Could not mount Google Drive. Aborting. Error: {e}"); return

    df = pd.read_csv(os.path.join(DATA_DIR, "nba_advanced_training_data.csv"))
    print(f"Loaded {len(df)} rows of advanced training data.")

    # --- Stage 1 & 2: Train Regression Models (as before) ---
    print("\n--- STAGE 1 & 2: Training Regression Models ---")
    min_features = [c for c in df.columns if 'ROLL_AVG_MIN' in c or 'TREND_MIN' in c] + ['DAYS_OF_REST']
    imputer_min = SimpleImputer(strategy='mean'); X_min_imputed = imputer_min.fit_transform(df[min_features])
    minutes_model = lgb.LGBMRegressor(objective='rmse', random_state=42); minutes_model.fit(X_min_imputed, df['TARGET_MIN'])
    joblib.dump(minutes_model, os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")); joblib.dump(imputer_min, os.path.join(MODEL_DIR, "minutes_imputer.pkl"))

    df['Proj_MIN'] = minutes_model.predict(X_min_imputed)
    stat_features = [c for c in df.columns if ('ROLL_AVG' in c or 'TREND' in c or 'OPP_DvT' in c) and 'MIN' not in c] + ['DAYS_OF_REST', 'Proj_MIN']
    imputer_stats = SimpleImputer(strategy='mean'); X_stats_imputed = imputer_stats.fit_transform(df[stat_features])
    joblib.dump(imputer_stats, os.path.join(MODEL_DIR, "stats_imputer.pkl"))

    stat_models = {};
    for prop in ['PTS', 'REB', 'AST']:
        model = lgb.LGBMRegressor(objective='rmse', random_state=42); model.fit(X_stats_imputed, df[f'TARGET_{prop}'])
        df[f'Proj_{prop}'] = model.predict(X_stats_imputed)
        joblib.dump(model, os.path.join(MODEL_DIR, f"{prop.lower()}_lgbm_model.pkl")); stat_models[prop] = model
    print("✅ Regression models for MIN, PTS, REB, AST trained and saved.")

    # --- NEW: STAGE 3: Training Classification Models (DD/TD) ---
    print("\n--- STAGE 3: Training Classification Models (DD/TD) ---")

    # The features for our classifier are the projections from the regression models
    classification_features = ['Proj_MIN', 'Proj_PTS', 'Proj_REB', 'Proj_AST']
    X_class = df[classification_features]

    for prop in ['DD', 'TD']:
        print(f"\n--- Training LGBM Classifier for: {prop} ---")
        target_col = f'TARGET_{prop}'
        y_class = df[target_col]

        # Check if we have enough positive samples to train
        if y_class.sum() < 50:
            print(f"  - Skipping {prop} model: Not enough positive samples ({y_class.sum()}) in the dataset.")
            continue

        X_train, X_test, y_train, y_test = train_test_split(X_class, y_class, test_size=0.2, random_state=42, stratify=y_class)

        # Use LGBMClassifier, with class_weight='balanced' for imbalanced data
        classifier = lgb.LGBMClassifier(objective='binary', class_weight='balanced', random_state=42)
        classifier.fit(X_train, y_train)

        # Evaluate with AUC, a proper metric for classification
        probs = classifier.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probs)
        print(f"  - Final Model AUC for {prop}: {auc:.3f}")

        joblib.dump(classifier, os.path.join(MODEL_DIR, f"{prop.lower()}_lgbm_classifier.pkl"))
        print(f"✅ Saved {prop} Classification Model.")

if __name__ == "__main__":
    train_and_evaluate_models_lgbm()

Mounting Google Drive...
Mounted at /content/drive
Loaded 225312 rows of advanced training data.

--- STAGE 1 & 2: Training Regression Models ---
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.019991 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1028
[LightGBM] [Info] Number of data points in the train set: 225312, number of used features: 5
[LightGBM] [Info] Start training from score 22.805143


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.038106 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3677
[LightGBM] [Info] Number of data points in the train set: 225312, number of used features: 17
[LightGBM] [Info] Start training from score 10.381347


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.041015 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3677
[LightGBM] [Info] Number of data points in the train set: 225312, number of used features: 17
[LightGBM] [Info] Start training from score 4.156010


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.047288 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3677
[LightGBM] [Info] Number of data points in the train set: 225312, number of used features: 17
[LightGBM] [Info] Start training from score 2.296274


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Regression models for MIN, PTS, REB, AST trained and saved.

--- STAGE 3: Training Classification Models (DD/TD) ---

--- Training LGBM Classifier for: DD ---
[LightGBM] [Info] Number of positive: 14932, number of negative: 165317
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014000 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 180249, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
  - Final Model AUC for DD: 0.897
✅ Saved DD Classification Model.

--- Training LGBM Classifier for: TD ---
[LightGBM] [Info] Number of positive: 842, number of negative: 179407
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008009 seconds.
You can set `force_col_wise=true` to remove the overhead.
[Li

"9b96859c5e0781bb4c5809215a5ed94b",
"f4cfcea7cc6120a97559f69d6049596e",
"7b9405e2febc561e554b7169a42cefd1",
"146dff42ae19a83208b059d4855095c9",
"f8474479351ccc685ccae9f4716c23c5",
"39756c093848b6968c899a4e1ded85f9",
"9705a1f6b609305047eb8ec9243cc54a",
"4217574ccdd90732483d98f24772f66b",
"66953d96fcb2ec72a47e7f39901258e7",
"94304f58fb95fd1d45159b9c8d292724",
"e384411e342cac5c9408804c622703f1",
"dab5c3da80b0415f64d4d94890974cdf",
"9705a1f6b609305047eb8ec9243cc54a",
"3aae94b51cc3dcb8e0623bf2633a695a",
"a4e4d3eef7d360055b96a1351560817f",
"bd6c88c66ff4c7c057aafaffcdd1b8d",
"1bfcac154fb2b490bf55a80a18275db1",
"7b64d450f91079f60211cb33033594b4",
"ec9dc1e1aa0543c1b3fa3d2576fabffd",
"6bbfe433274aef6fee5aa5543ad69055",
"1f5e9cdc190854c605b533917b2c9b82",
"858b7234b5f77661345f205b1c5ec121",
"39756c093848b6968c899a4e1ded85f9",
"21caebe8058c338a7ba76f6c9fb550cb",
"94304f58fb95fd1d45159b9c8d292724",

Script 3

In [ ]:
# ==============================================================================
# SCRIPT 3: Definitive Prediction Engine (Final Production Version, 2025–26)
# READY-TO-RUN (Google Colab) — uses models in:
# /content/drive/MyDrive/NBA/Training Model Advanced/
# ==============================================================================

# Install required packages (Colab)
!pip install -q xlsxwriter nba_api scikit-learn pandas joblib pytz unidecode requests lightgbm python-dateutil tqdm openpyxl

# --- Imports ---
import os
import re
import time
import joblib
import warnings
import requests
import numpy as np
import pandas as pd
import pytz
from datetime import datetime
from dateutil import parser
from unidecode import unidecode
from tqdm import tqdm
from google.colab import drive
from IPython.display import display
from nba_api.stats.endpoints import playergamelog, leaguedashplayerstats, playergamelogs
from nba_api.stats.static import teams

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------------
# CONFIG
# ---------------------
MODEL_DIR = "/content/drive/MyDrive/NBA/Training Model Advanced/"
DATA_DIR = "/content/drive/MyDrive/NBA/Historical Data/"
OUTPUT_DIR = "/content/drive/MyDrive/NBA/Predictions/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
ROLLING_WINDOWS = [5, 10, 20]
MARKETS = {
    'player_points': 'PTS', 'player_rebounds': 'REB', 'player_assists': 'AST',
    'player_points_rebounds_assists': 'PRA', 'player_points_rebounds': 'P+R',
    'player_points_assists': 'P+A', 'player_rebounds_assists': 'R+A',
    'player_double_double': 'DD', 'player_triple_double': 'TD'
}
ODDS_API_KEYS = [
    # Keep your keys here (some sample keys included earlier)
    "9b96859c5e0781bb4c5809215a5ed94b",
    "f4cfcea7cc6120a97559f69d6049596e",
    "7b9405e2febc561e554b7169a42cefd1",
    "146dff42ae19a83208b059d4855095c9"
]

# simple cache for fetched features
feature_cache = {}

# ---------------------
# UTILITIES
# ---------------------
def normalize_player_name(name):
    if not isinstance(name, str):
        return None
    name = unidecode(name.lower())
    name = re.sub(r'\s+(jr|sr|ii|iii|iv)\.?$', '', name)
    name = re.sub(r'[^\w\s]', '', name)
    return re.sub(r'\s+', ' ', name).strip()

def get_current_nba_season(live_date):
    year, month = live_date.year, live_date.month
    return f"{year}-{str(year+1)[-2:]}" if month >= 9 else f"{year-1}-{str(year)[-2:]}"

def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def decimal_to_american(decimal_odds):
    if pd.isna(decimal_odds) or decimal_odds is None:
        return None
    try:
        d = float(decimal_odds)
    except Exception:
        return None
    if d >= 2.0:
        return f"+{int((d - 1) * 100)}"
    # d < 2 implies favorite; compute approximate American
    try:
        return f"-{int(100 / (d - 1))}"
    except Exception:
        return None

# ---------------------
# DATA / FEATURE FUNCTIONS
# ---------------------
def fetch_injury_data():
    """Fetch injuries from ESPN; return dict normalized_name -> status"""
    try:
        r = requests.get("https://site.api.espn.com/apis/site/v2/sports/basketball/nba/injuries", timeout=20)
        r.raise_for_status()
        data = r.json()
        inj = {}
        for grouping in data.get("injuries", []):
            for item in grouping.get("injuries", []):
                name = item['athlete'].get('displayName')
                status = item.get('type', {}).get('description') or ''
                inj[normalize_player_name(name)] = status.title()
        return inj
    except Exception:
        return {}

def calculate_live_dvt(season, data_dir, player_trait_map):
    """
    Calculate Defense-vs-Trait (DvT). If no trait map provided, return empty DataFrame.
    This function expects player_trait_map with columns PLAYER_ID (int) and TRAIT (int/str).
    """
    if player_trait_map is None or player_trait_map.empty:
        # return empty DataFrame with no columns (caller should handle)
        return pd.DataFrame()

    gamelog_cache = os.path.join(data_dir, f"live_gamelogs_{season}.csv")
    if os.path.exists(gamelog_cache):
        df = pd.read_csv(gamelog_cache, low_memory=False)
    else:
        try:
            df = playergamelogs.PlayerGameLogs(season_nullable=season, timeout=120).get_data_frames()[0]
            df.to_csv(gamelog_cache, index=False)
        except Exception:
            return pd.DataFrame()

    df['PLAYER_ID'] = pd.to_numeric(df['PLAYER_ID'], errors='coerce')
    player_trait_map['PLAYER_ID'] = pd.to_numeric(player_trait_map['PLAYER_ID'], errors='coerce')
    merged = pd.merge(df, player_trait_map, on='PLAYER_ID', how='left').dropna(subset=['TRAIT'])
    if merged.empty:
        return pd.DataFrame()
    merged['OPPONENT_ABBR'] = merged['TEAM_ABBREVIATION']  # simplified
    stats = merged.groupby(['OPPONENT_ABBR', 'TRAIT'])[['PTS', 'REB', 'AST']].mean().reset_index()
    pivot = stats.pivot_table(index='OPPONENT_ABBR', columns='TRAIT', values=['PTS', 'REB', 'AST'])
    pivot.columns = [f'OPP_DvT_{stat}_vs_{int(trait)}' for stat, trait in pivot.columns]
    return pivot

def get_live_features_for_player(player_id, current_season, previous_season):
    """
    Returns a dict of rolling averages and trends for the player, cached by player-season.
    Uses nba_api PlayerGameLog.
    """
    cache_key = f"{player_id}_{current_season}"
    if cache_key in feature_cache:
        return feature_cache[cache_key]

    try:
        cur = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, timeout=60).get_data_frames()[0]
        prev = playergamelog.PlayerGameLog(player_id=player_id, season=previous_season, timeout=60).get_data_frames()[0]
    except Exception:
        return None

    logs = pd.concat([prev, cur], ignore_index=True)
    if logs.empty:
        return None

    # ensure numeric
    for col in ['PTS', 'REB', 'AST', 'MIN']:
        if col not in logs.columns:
            logs[col] = np.nan
    logs['GAME_DATE'] = pd.to_datetime(logs['GAME_DATE'], errors='coerce')
    logs = logs.sort_values('GAME_DATE')
    features = {}
    for stat in ['PTS', 'REB', 'AST', 'MIN']:
        for w in ROLLING_WINDOWS:
            features[f'ROLL_AVG_{stat}_{w}'] = logs[stat].tail(w).astype(float).mean() if not logs[stat].tail(w).empty else np.nan
        # trend: short - long
        features[f'TREND_{stat}'] = features.get(f'ROLL_AVG_{stat}_{ROLLING_WINDOWS[0]}', np.nan) - features.get(f'ROLL_AVG_{stat}_{ROLLING_WINDOWS[-1]}', np.nan)
    last_game = logs['GAME_DATE'].dropna()
    if last_game.empty:
        features['DAYS_OF_REST'] = np.nan
    else:
        last_game = last_game.max()
        features['DAYS_OF_REST'] = (datetime.utcnow() - last_game.to_pydatetime()).days
    feature_cache[cache_key] = features
    return features

# ---------------------
# ODDS / KEY HELPERS
# ---------------------
def check_and_select_key(live_today_str, eastern_tz):
    """Return a working key and list of events for today (odds API)."""
    valid_keys = []
    key_audit = {}
    for key in ODDS_API_KEYS:
        try:
            r = requests.get(f'https://api.the-odds-api.com/v4/sports?apiKey={key}', timeout=8)
            if r.status_code == 200:
                remaining = int(r.headers.get('x-requests-remaining', 0) or 0)
                valid_keys.append(key)
                key_audit[key] = remaining
        except Exception:
            continue
    if not valid_keys:
        return None, None, None
    # pick first key with any allowance
    for k in valid_keys:
        if key_audit.get(k, 0) >= 0:
            schedule_key = k
            break
    try:
        params = {'apiKey': schedule_key}
        resp = requests.get('https://api.the-odds-api.com/v4/sports/basketball_nba/events', params=params, timeout=20)
        resp.raise_for_status()
        events = resp.json()
    except Exception:
        return schedule_key, [], 0

    events_today = [g for g in events if parser.isoparse(g["commence_time"]).astimezone(eastern_tz).date().isoformat() == live_today_str]
    return schedule_key, events_today, key_audit.get(schedule_key, None)

# ---------------------
# MAIN RUN
# ---------------------
def run_daily_projections_advanced():
    # Mount Drive (Colab)
    try:
        drive.mount('/content/drive', force_remount=True)
    except Exception:
        pass

    # Validate model dir
    if not os.path.exists(MODEL_DIR):
        raise FileNotFoundError(f"MODEL_DIR not found: {MODEL_DIR}")

    # Load models that exist in your drive
    models = {}
    imputers = {}
    # Load AST model
    ast_path = os.path.join(MODEL_DIR, "ast_lgbm_model.pkl")
    minutes_path = os.path.join(MODEL_DIR, "minutes_lgbm_model.pkl")
    dd_path = os.path.join(MODEL_DIR, "dd_lgbm_classifier.pkl")
    td_path = os.path.join(MODEL_DIR, "td_lgbm_classifier.pkl")
    minutes_imputer_path = os.path.join(MODEL_DIR, "minutes_imputer.pkl")
    stats_imputer_path = os.path.join(MODEL_DIR, "stats_imputer.pkl")

    # defensive loads
    try:
        models['AST'] = joblib.load(ast_path) if os.path.exists(ast_path) else None
        models['MIN'] = joblib.load(minutes_path) if os.path.exists(minutes_path) else None
        models['DD']  = joblib.load(dd_path) if os.path.exists(dd_path) else None
        models['TD']  = joblib.load(td_path) if os.path.exists(td_path) else None
        imputers['MIN']   = joblib.load(minutes_imputer_path) if os.path.exists(minutes_imputer_path) else None
        imputers['STATS'] = joblib.load(stats_imputer_path) if os.path.exists(stats_imputer_path) else None
    except Exception as e:
        raise RuntimeError(f"Error loading model files: {e}")

    # Context
    eastern = pytz.timezone('America/New_York')
    live_now = datetime.now(eastern)
    live_today_str = live_now.date().isoformat()
    season = get_current_nba_season(live_now)
    prev_season = f"{int(season[:4]) - 1}-{season[:2]}"

    print(f"Running projections for {live_today_str} (Season: {season})")
    injury_data = fetch_injury_data()

    # Choose API key and fetch today's events
    working_key, events_today, requests_remaining = check_and_select_key(live_today_str, eastern)
    if working_key is None:
        print("No working odds API key found. Continuing — will only produce projections (no odds).")

    # Create a minimal player_trait_map if none exists in DATA_DIR
    trait_map_path = os.path.join(DATA_DIR, "player_trait_map.csv")
    if os.path.exists(trait_map_path):
        player_trait_map = pd.read_csv(trait_map_path)
    else:
        player_trait_map = pd.DataFrame(columns=["PLAYER_ID", "TRAIT"])

    dvt_df = calculate_live_dvt(season, DATA_DIR, player_trait_map)

    # Fetch player props from odds API (if available)
    props = []
    if events_today:
        for event in events_today:
            event_id = event.get('id') or event.get('event_id') or event.get('id', None)
            if not event_id:
                continue
            params = {'apiKey': working_key, 'regions': 'us', 'markets': ','.join(MARKETS.keys()), 'bookmakers': 'draftkings,fanduel'}
            try:
                r = requests.get(f"https://api.the-odds-api.com/v4/sports/basketball_nba/events/{event_id}/odds", params=params, timeout=20)
                r.raise_for_status()
                payload = r.json()
                for book in payload.get('bookmakers', []):
                    for market in book.get('markets', []):
                        mkey = market.get('key')
                        prop_short = MARKETS.get(mkey)
                        if not prop_short:
                            continue
                        for outcome in market.get('outcomes', []):
                            desc = outcome.get('description') or outcome.get('name') or ''
                            line = outcome.get('point')
                            price = outcome.get('price') or outcome.get('decimal') or outcome.get('odds')
                            name = outcome.get('name') or outcome.get('label')
                            props.append({
                                'event_id': event_id,
                                'Game': f"{event.get('away_team')} @ {event.get('home_team')}",
                                'Player': desc,
                                'Prop': prop_short,
                                'Line': safe_float(line),
                                'Decimal_Odds': safe_float(price),
                                'OutcomeName': name,
                                'Bookmaker': book.get('title')
                            })
            except Exception:
                continue

    props_df = pd.DataFrame(props)
    if props_df.empty:
        print("No props returned from odds API for today (or no odds available). The script will still generate player projections without odds.")

    # Build projections for players that appear in props OR for all players (depending on preference)
    # We'll base on players returned by leaguedashplayerstats and join by normalized name.
    try:
        players_df = leaguedashplayerstats.LeagueDashPlayerStats(season=season, timeout=60).get_data_frames()[0]
    except Exception as e:
        raise RuntimeError(f"Could not fetch player list from nba_api: {e}")

    players_df['match_key'] = players_df['PLAYER_NAME'].apply(normalize_player_name)
    if not props_df.empty:
        props_df['match_key'] = props_df['Player'].apply(normalize_player_name)
        to_project = players_df[players_df['match_key'].isin(props_df['match_key'].unique())].copy()
        if to_project.empty:
            # fallback: project all players
            to_project = players_df.copy()
    else:
        to_project = players_df.copy()

    print(f"Preparing to project {len(to_project)} players...")

    projection_rows = []
    for _, row in tqdm(to_project.iterrows(), total=len(to_project)):
        pid = row.get('PLAYER_ID')
        pname = row.get('PLAYER_NAME')
        mk = row.get('match_key')

        # fetch live rolling features
        try:
            features = get_live_features_for_player(int(pid), season, prev_season)
        except Exception:
            features = None
        if features is None:
            # try to derive simple features from leaguedash (fallback)
            features = {}
            # try to use per-game stats columns if present
            for stat in ['PTS','REB','AST','MIN']:
                col = f'PTS' if stat == 'PTS' else stat
                features[f'ROLL_AVG_{stat}_5'] = np.nan
            features['DAYS_OF_REST'] = np.nan

        # build model input
        model_input_df = pd.DataFrame([features])

        # Predictions
        pred_pts = features.get('ROLL_AVG_PTS_5', np.nan)
        pred_reb = features.get('ROLL_AVG_REB_5', np.nan)
        pred_ast = np.nan
        pred_min = np.nan
        dd_prob = np.nan
        td_prob = np.nan

        try:
            if models.get('AST') is not None:
                pred_ast = float(models['AST'].predict(model_input_df.fillna(0))[0])
        except Exception:
            pred_ast = features.get('ROLL_AVG_AST_5', np.nan)

        try:
            if models.get('MIN') is not None:
                pred_min = float(models['MIN'].predict(model_input_df.fillna(0))[0])
        except Exception:
            pred_min = features.get('ROLL_AVG_MIN_5', np.nan)

        try:
            if models.get('DD') is not None:
                if hasattr(models['DD'], "predict_proba"):
                    dd_prob = float(models['DD'].predict_proba(model_input_df.fillna(0))[0][1])
                else:
                    dd_prob = float(models['DD'].predict(model_input_df.fillna(0))[0])
        except Exception:
            dd_prob = np.nan

        try:
            if models.get('TD') is not None:
                if hasattr(models['TD'], "predict_proba"):
                    td_prob = float(models['TD'].predict_proba(model_input_df.fillna(0))[0][1])
                else:
                    td_prob = float(models['TD'].predict(model_input_df.fillna(0))[0])
        except Exception:
            td_prob = np.nan

        # If PTS/REB couldn't be modeled, fall back to rolling averages
        if np.isnan(pred_pts):
            pred_pts = features.get('ROLL_AVG_PTS_5', np.nan)
        if np.isnan(pred_reb):
            pred_reb = features.get('ROLL_AVG_REB_5', np.nan)

        pra = np.nansum([pred_pts, pred_reb, pred_ast])

        projection_rows.append({
            'PLAYER_ID': pid,
            'Player': pname,
            'match_key': mk,
            'Pred_PTS': pred_pts,
            'Pred_REB': pred_reb,
            'Pred_AST': pred_ast,
            'Pred_MIN': pred_min,
            'Pred_PRA': pra,
            'Pred_DD_prob': dd_prob,
            'Pred_TD_prob': td_prob,
            'Injury': injury_data.get(mk, '')
        })

    proj_df = pd.DataFrame(projection_rows)
    if proj_df.empty:
        print("No projections generated.")
        return proj_df

    # Merge projections with odds (if available)
    if not props_df.empty:
        merged = pd.merge(props_df, proj_df, on='match_key', how='left', suffixes=('_odds',''))
    else:
        # create a minimal odds-like df just to format output
        merged = proj_df.copy()
        merged['Game'] = np.nan
        merged['Prop'] = np.nan
        merged['Line'] = np.nan
        merged['Decimal_Odds'] = np.nan
        merged['Bookmaker'] = np.nan

    # Normalize numeric columns and compute Edge
    merged['Projection'] = merged.apply(
        lambda r: r['Pred_PTS'] if r['Prop'] == 'PTS' else
                  (r['Pred_REB'] if r['Prop'] == 'REB' else
                   (r['Pred_AST'] if r['Prop'] == 'AST' else
                    (r['Pred_PRA'] if r['Prop'] == 'PRA' else np.nan)))
        , axis=1
    )

    merged['Line_Num'] = pd.to_numeric(merged['Line'], errors='coerce')
    merged['Projection_Num'] = pd.to_numeric(merged['Projection'], errors='coerce')

    # For Over/Under props (numeric line)
    ou_mask = merged['Line_Num'].notna() & merged['Projection_Num'].notna()
    merged.loc[ou_mask, 'Edge'] = ((merged.loc[ou_mask, 'Projection_Num'] - merged.loc[ou_mask, 'Line_Num']) / merged.loc[ou_mask, 'Line_Num']) * 100

    # For DD/TD: compare model probability to implied decimal odds (if provided)
    if 'Decimal_Odds' in merged.columns:
        merged['Implied_Prob_Yes'] = merged['Decimal_Odds'].apply(lambda x: (1.0 / x) if (not pd.isna(x) and x > 0) else np.nan)
    else:
        merged['Implied_Prob_Yes'] = np.nan

    yn_mask = merged['Prop'].isin(['DD','TD']) & merged['Pred_DD_prob'].notna() | merged['Prop'].isin(['DD','TD']) & merged['Pred_TD_prob'].notna()
    # Make safe: for DD use Pred_DD_prob, for TD use Pred_TD_prob
    merged['Projection_Prob'] = np.nan
    merged.loc[merged['Prop'] == 'DD', 'Projection_Prob'] = merged['Pred_DD_prob']
    merged.loc[merged['Prop'] == 'TD', 'Projection_Prob'] = merged['Pred_TD_prob']
    yn_mask = merged['Projection_Prob'].notna() & merged['Implied_Prob_Yes'].notna()
    merged.loc[yn_mask, 'Edge'] = (merged.loc[yn_mask, 'Projection_Prob'] * 100) - (merged.loc[yn_mask, 'Implied_Prob_Yes'] * 100)

    # Assign Bets
    merged['Bet'] = 'Pass'
    merged.loc[ou_mask, 'Bet'] = np.where(merged.loc[ou_mask, 'Edge'] > 10, 'Over',
                                          np.where(merged.loc[ou_mask, 'Edge'] < -10, 'Under', 'Pass'))
    # Only bet Yes on DD/TD when edge > 5%
    merged.loc[merged['Prop'].isin(['DD','TD']), 'Bet'] = np.where(merged.loc[merged['Prop'].isin(['DD','TD']), 'Edge'] > 5, 'Yes', 'Pass')

    # Display & export
    out_cols = ['Game','Player','Prop','Line','Decimal_Odds','Bookmaker','Projection_Num','Edge','Bet','Injury']
    # ensure columns exist
    for c in out_cols:
        if c not in merged.columns:
            merged[c] = np.nan

    # Round numeric fields
    merged['Projection_Num'] = merged['Projection_Num'].round(2)
    merged['Edge'] = merged['Edge'].round(1)

    # Save to Excel
    out_file = os.path.join(OUTPUT_DIR, f"{live_now.strftime('%m-%d-%Y')} NBA Advanced Props.xlsx")
    try:
        merged.to_excel(out_file, index=False)
        print(f"\n✅ Exported results to: {out_file}")
    except Exception as e:
        print(f"Warning: could not write excel file: {e}")

    # Show summary of +EV bets
    ev = merged[merged['Bet'] != 'Pass'].copy()
    if not ev.empty:
        ev = ev.sort_values(by='Edge', ascending=False)
        display(ev[out_cols].head(200))
    else:
        print("No EV bets found for current thresholds. Showing top projections:")
        display(merged.sort_values(by='Projection_Num', ascending=False).head(30))

    # Print API usage if available
    if requests_remaining is not None:
        print(f"API requests remaining on selected key: {requests_remaining}")

    return merged

# ---------------------
# Run script
# ---------------------
if __name__ == "__main__":
    pd.set_option('display.max_rows', 200)
    pd.set_option('display.width', 1200)
    result = run_daily_projections_advanced()
    # result is a DataFrame with merged projections and odds (if available)


Mounted at /content/drive
Running projections for 2025-11-11 (Season: 2025-26)
No props returned from odds API for today (or no odds available). The script will still generate player projections without odds.
Preparing to project 454 players...


  0%|          | 0/454 [00:00<?, ?it/s]/tmp/ipython-input-390323708.py:179: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  features['DAYS_OF_REST'] = (datetime.utcnow() - last_game.to_pydatetime()).days
  0%|          | 1/454 [00:01<14:49,  1.96s/it]/tmp/ipython-input-390323708.py:179: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  features['DAYS_OF_REST'] = (datetime.utcnow() - last_game.to_pydatetime()).days
  0%|          | 2/454 [00:03<13:50,  1.84s/it]/tmp/ipython-input-390323708.py:179: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.no


✅ Exported results to: /content/drive/MyDrive/NBA/Predictions/11-11-2025 NBA Advanced Props.xlsx
No EV bets found for current thresholds. Showing top projections:


,PLAYER_ID,Player,match_key,Pred_PTS,Pred_REB,Pred_AST,Pred_MIN,Pred_PRA,Pred_DD_prob,Pred_TD_prob,Injury,Game,Prop,Line,Decimal_Odds,Bookmaker,Projection,Line_Num,Projection_Num,Edge,Implied_Prob_Yes,Projection_Prob,Bet
0,1631260,AJ Green,aj green,8.800000,2.400000,2.180413,26.800000,13.380413,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
1,1642358,AJ Johnson,aj johnson,1.400000,0.600000,0.654298,5.800000,2.654298,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
2,203932,Aaron Gordon,aaron gordon,17.000000,6.000000,1.536801,28.000000,24.536801,NaN,NaN,Day-To-Day,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
3,1628988,Aaron Holiday,aaron holiday,3.250000,0.750000,1.068968,6.750000,5.068968,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
4,1630174,Aaron Nesmith,aaron nesmith,20.000000,4.600000,1.208714,30.400000,25.808714,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
5,1630598,Aaron Wiggins,aaron wiggins,16.000000,3.000000,1.994601,27.400000,20.994601,NaN,NaN,Out,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
6,1642846,Ace Bailey,ace bailey,8.200000,3.800000,1.702623,21.600000,13.702623,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
7,1641737,Adem Bona,adem bona,2.000000,3.400000,0.845971,14.400000,6.245971,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
8,1642349,Ajay Mitchell,ajay mitchell,15.600000,4.800000,4.319357,28.800000,24.719357,NaN,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass
9,201143,Al Horford,al horford,4.200000,4.600000,1.393153,20.000000,10.193153,NaN,NaN,Out,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Pass


API requests remaining on selected key: 0


In [ ]:
# ==============================================================================
# NBA NORMALIZATION & MATCHUP OUTPUT (2025–26+ Optimized Version)
# ==============================================================================
# Required installs
!pip install pandas requests rapidfuzz openpyxl xlsxwriter

import pandas as pd
import requests
from rapidfuzz import process, fuzz
from io import BytesIO

# ==============================================================================
# STEP 1: FETCH ESPN TEAMS AND ROSTERS
# ==============================================================================
def fetch_espn_teams():
    url = "http://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
    response = requests.get(url).json()
    teams = {}
    for t in response["sports"][0]["leagues"][0]["teams"]:
        team = t["team"]
        teams[team["displayName"]] = {
            "id": team["id"],
            "abbrev": team.get("abbreviation", ""),
            "slug": team.get("slug", ""),
        }
    return teams

def fetch_espn_roster(team_id):
    url = f"https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/roster"
    response = requests.get(url).json()
    players = []
    for athlete in response.get("athletes", []):
        players.append({
            "player_id": athlete["athlete"]["id"],
            "player_name": athlete["athlete"]["displayName"],
            "team_id": team_id,
            "injury_status": athlete.get("injuryStatus", "Healthy")
        })
    return players

espn_teams = fetch_espn_teams()
all_players = []
for tname, tmeta in espn_teams.items():
    all_players.extend(fetch_espn_roster(tmeta["id"]))
espn_roster_df = pd.DataFrame(all_players)

# ==============================================================================
# STEP 2: FUZZY MATCH FUNCTIONS
# ==============================================================================
def get_team_id_from_name(odds_team_name, espn_team_dict):
    # Exact match first
    for k, v in espn_team_dict.items():
        if k.lower() == odds_team_name.lower():
            return v["id"]
    # Fuzzy fallback
    match, score = process.extractOne(odds_team_name, espn_team_dict.keys(), scorer=fuzz.token_sort_ratio)
    if score > 85:
        return espn_team_dict[match]["id"]
    print(f"⚠️ Team unmatched: {odds_team_name}")
    return None

def match_player_by_team(odds_player_name, team_name, espn_roster_df, espn_teams):
    team_id = get_team_id_from_name(team_name, espn_teams)
    if not team_id:
        return None, None, None
    roster = espn_roster_df[espn_roster_df["team_id"] == team_id]
    if roster.empty:
        return None, None, None
    match, score = process.extractOne(odds_player_name, roster["player_name"], scorer=fuzz.token_sort_ratio)
    if match and score > 85:
        row = roster[roster["player_name"] == match].iloc[0]
        return row["player_id"], row["player_name"], row["injury_status"]
    print(f"⚠️ Player unmatched: {odds_player_name} ({team_name})")
    return None, None, None

# ==============================================================================
# STEP 3: NORMALIZE A SINGLE EVENT
# ==============================================================================
def normalize_event(event, espn_teams, espn_roster_df):
    home_id = get_team_id_from_name(event["home_team"], espn_teams)
    away_id = get_team_id_from_name(event["away_team"], espn_teams)

    normalized_players = []
    for p in event["players"]:
        pid, pname, inj = match_player_by_team(p["name"], p["team"], espn_roster_df, espn_teams)
        normalized_players.append({
            "matchup": f"{event['away_team']} @ {event['home_team']}",
            "event_id": event["id"],
            "team": p["team"],
            "team_id": get_team_id_from_name(p["team"], espn_teams),
            "player_id": pid,
            "player_name": pname,
            "injury_status": inj
        })
    return normalized_players

# ==============================================================================
# STEP 4: DEMO EVENT (You’ll replace this with your live Odds feed)
# ==============================================================================
demo_event = {
    "id": "GAME_001",
    "home_team": "Dallas Mavericks",
    "away_team": "Los Angeles Lakers",
    "players": [
        {"name": "Luka Doncic", "team": "Dallas Mavericks"},
        {"name": "LeBron James", "team": "Los Angeles Lakers"},
        {"name": "Anthony Davis", "team": "Los Angeles Lakers"},
    ]
}

events = [demo_event]  # could be many from your odds feed
records = []
for e in events:
    records.extend(normalize_event(e, espn_teams, espn_roster_df))
df = pd.DataFrame(records)

# ==============================================================================
# STEP 5: CLEAN, GROUP, AND CENTER FORMAT IN EXCEL
# ==============================================================================
df.sort_values(["matchup", "team", "player_name"], inplace=True)

output_path = "/content/NBA_Normalized_Matchups.xlsx"
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    df.to_excel(writer, index=False, sheet_name="Normalized")
    workbook = writer.book
    worksheet = writer.sheets["Normalized"]

    # Center all cells
    center_fmt = workbook.add_format({"align": "center", "valign": "vcenter"})
    worksheet.set_column(0, len(df.columns) - 1, 22, center_fmt)

    # Add autofilter and freeze header
    worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
    worksheet.freeze_panes(1, 0)

print(f"✅ Normalized and grouped Excel saved to: {output_path}")